# Descarga y procesamiento de datos climatológicos de CONAGUA

**Proyecto:** Análisis de la producción agrícola en México

**Fuente:** Comisión Nacional del Agua (CONAGUA) /
Servicio Meteorológico Nacional (SMN).

**Periodo climático:** 1981–2022.

## Objetivo

Construir un dataset climatológico nacional utilizando
estaciones distribuidas espacialmente, evitando la
redundancia de estaciones cercanas y excluyendo islas.

La adquisición y transformación de datos se realizará
mediante las funciones del paquete `proyecto_agronomia`.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
import requests
import numpy as np

from proyecto_agronomia.config import (
    CONAGUA_KMZ_PATH,
    CONAGUA_PROCESSED_DIR,
    CONAGUA_ANIO_INICIAL,
    CONAGUA_ANIO_FINAL,
    CONAGUA_DISTANCIA_MIN_KM,
    CONAGUA_MEXICO_GEOJSON_URL,
    CONAGUA_RAW_DIR,
)
from proyecto_agronomia.datasets.clima_con import (
    descargar_catalogo_conagua,
    descargar_estaciones_conagua,
    leer_catalogo_conagua,
    filtrar_estaciones_continentales,
    seleccionar_estaciones_espaciales,
    crear_mapa_estaciones_conagua,
    convertir_conagua_nacional,
)

2026-10-27 14:40:06.415 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## 1. Obtención del catálogo de estaciones

Se descarga el catálogo nacional de estaciones climatológicas
del SMN en formato KMZ.

Posteriormente se extraen los metadatos de cada estación,
incluyendo ubicación, periodo de registro y URL mensual.

In [2]:
ruta_kmz = descargar_catalogo_conagua()

df_estaciones = leer_catalogo_conagua(
    ruta_kmz
)

print("Estaciones:", len(df_estaciones))
print("Estados:", df_estaciones["estado"].nunique())

display(df_estaciones.head())

2026-10-27 14:40:07.253 | INFO     | proyecto_agronomia.datasets.clima_con:descargar_catalogo_conagua:66 - El catálogo KMZ de CONAGUA ya existe.
2026-10-27 14:40:08.133 | SUCCESS  | proyecto_agronomia.datasets.clima_con:leer_catalogo_conagua:387 - Catálogo CONAGUA leído: 5,526 estaciones.
Estaciones: 5526
Estados: 32


,estacion,nombre,estado,municipio,situacion,inicio,mas_reciente,latitud,longitud,altitud,url_mensual
0,10001,ATOTONILCO,DURANGO,CUENCAMÉ,OPERANDO,1969-03-01,2026-04-30,24.640556,-103.698333,1950.0,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
1,10003,CANELAS,DURANGO,CANELAS,OPERANDO,1960-11-01,2026-03-31,25.124844,-106.547422,1754.0,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
2,10004,CAÑON FERNANDEZ,DURANGO,CUENCAMÉ,OPERANDO,1944-02-01,2026-07-31,25.265278,-103.773611,1200.0,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
3,10005,CEBALLOS,DURANGO,MAPIMÍ,OPERANDO,1956-02-01,2026-01-31,26.323889,-104.351111,1188.0,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
4,10006,CENDRADILLAS,DURANGO,GUANACEVÍ,OPERANDO,1961-01-01,2020-04-30,26.282778,-106.010556,2270.0,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...


In [3]:
resumen_estados = (
    df_estaciones
    .groupby("estado")
    .agg(
        estaciones=("estacion", "nunique"),
        municipios=("municipio", "nunique"),
    )
    .reset_index()
    .sort_values(
        "estaciones",
        ascending=False,
    )
)

display(resumen_estados)

,estado,estaciones,municipios
29,VERACRUZ DE IGNACIO DE LA LLAVE,379,147
19,OAXACA,357,208
16,MÉXICO,353,99
4,CHIAPAS,303,95
25,SONORA,289,68
5,CHIHUAHUA,279,60
13,JALISCO,275,121
14,MICHOACÁN DE OCAMPO,241,91
11,GUERRERO,238,73
20,PUEBLA,215,116


In [4]:
from loguru import logger

def filtrar_estaciones_periodo(
    estaciones: pd.DataFrame,
    anio_inicial: int = CONAGUA_ANIO_INICIAL,
    anio_final: int = CONAGUA_ANIO_FINAL,
) -> pd.DataFrame:
    """
    Selecciona estaciones cuyo periodo de registro se
    superpone con el intervalo de estudio.

    No exige que la estación esté operando actualmente.

    Este filtro no garantiza que todos los meses
    tengan observaciones disponibles.
    """

    if anio_inicial > anio_final:
        raise ValueError(
            "El año inicial no puede superar al año final."
        )

    inicio = pd.Timestamp(f"{anio_inicial}-01-01")
    fin = pd.Timestamp(f"{anio_final}-12-31")

    df = estaciones.copy()

    df = df[
        (df["inicio"] <= fin)
        & (df["mas_reciente"] >= inicio)
        & df["url_mensual"].notna()
    ].copy()

    df = df.reset_index(drop=True)

    logger.success(
        f"Filtro temporal {anio_inicial}-{anio_final}: "
        f"{len(df):,} estaciones candidatas."
    )

    return df

## 2. Filtrado temporal de estaciones

Se seleccionan estaciones cuyo periodo de registro se
superpone con 1981–2022.

Se incluyen estaciones activas e inactivas.

La selección es preliminar: posteriormente se evaluará
la cobertura efectiva de sus observaciones mensuales.

In [5]:
df_candidatas = filtrar_estaciones_periodo(
    df_estaciones
)

print("Estaciones originales:", len(df_estaciones))
print("Estaciones candidatas:", len(df_candidatas))
print(
    "Estaciones excluidas:",
    len(df_estaciones) - len(df_candidatas)
)

display(
    df_candidatas.groupby("estado")
    .agg(
        estaciones=("estacion", "nunique"),
        municipios=("municipio", "nunique")
    )
    .reset_index()
    .sort_values("estaciones", ascending=False)
)

2026-10-27 14:40:08.307 | SUCCESS  | __main__:filtrar_estaciones_periodo:36 - Filtro temporal 1981-2025: 4,980 estaciones candidatas.
Estaciones originales: 5526
Estaciones candidatas: 4980
Estaciones excluidas: 546


,estado,estaciones,municipios
16,MÉXICO,342,99
19,OAXACA,322,199
29,VERACRUZ DE IGNACIO DE LA LLAVE,314,138
4,CHIAPAS,266,92
13,JALISCO,252,121
25,SONORA,243,68
5,CHIHUAHUA,227,58
11,GUERRERO,222,72
14,MICHOACÁN DE OCAMPO,218,90
20,PUEBLA,198,111


In [6]:
df_continentales = filtrar_estaciones_continentales(
    df_candidatas
)

print("Estaciones candidatas:", len(df_candidatas))
print("Estaciones continentales:", len(df_continentales))
print(
    "Estaciones excluidas:",
    len(df_candidatas) - len(df_continentales)
)

2026-10-27 14:40:08.356 | INFO     | proyecto_agronomia.datasets.clima_con:obtener_territorio_continental:423 - Descargando límites geográficos de México...
2026-10-27 14:40:16.350 | SUCCESS  | proyecto_agronomia.datasets.clima_con:obtener_territorio_continental:479 - GeoJSON descargado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\clima\conagua\limites_mexico.geojson
2026-10-27 14:40:17.589 | SUCCESS  | proyecto_agronomia.datasets.clima_con:obtener_territorio_continental:532 - Geometría continental de México obtenida.
2026-10-27 14:40:23.269 | SUCCESS  | proyecto_agronomia.datasets.clima_con:filtrar_estaciones_continentales:590 - Filtro continental: 4,967 estaciones conservadas de 4,980.
Estaciones candidatas: 4980
Estaciones continentales: 4967
Estaciones excluidas: 13


In [7]:
claves_conservadas = set(
    df_continentales["estacion"]
)

df_excluidas = df_candidatas[
    ~df_candidatas["estacion"].isin(
        claves_conservadas
    )
]

display(
    df_excluidas[
        [
            "estacion",
            "nombre",
            "estado",
            "municipio",
            "latitud",
            "longitud",
        ]
    ]
)

,estacion,nombre,estado,municipio,latitud,longitud
1431,23009,ISLA HOLBOX,QUINTANA ROO,LÁZARO CÁRDENAS,21.521389,-87.372778
1432,23010,ISLA MUJERES,QUINTANA ROO,ISLA MUJERES,21.235556,-86.735000
1453,23048,COZUMEL (DGE),QUINTANA ROO,COZUMEL,20.510000,-86.946111
3452,18015,ISLA MARIA MADRE (OBS),NAYARIT,SAN BLAS,21.632778,-106.539167
3668,2026,ISLA GUADALUPE (OBS),BAJA CALIFORNIA,INSULAR - SIN MUNICIPIO,29.020000,-118.278056
3674,2027,ISLA CEDROS,BAJA CALIFORNIA,INSULAR - SIN MUNICIPIO,28.135000,-115.175000
3913,23005,COZUMEL (OBS),QUINTANA ROO,COZUMEL,20.416667,-86.950000
4575,4005,CAYO ARCAS,CAMPECHE,CAMPECHE,20.204722,-91.962222
4576,4006,CAYO ARENAS,YUCATÁN,PROGRESO,22.115000,-91.397778
4578,4016,ISLA PEREZ,CAMPECHE,INSULAR - SIN MUNICIPIO,22.382500,-89.687778


## 4. Selección espacial de estaciones climatológicas

Se utiliza una estrategia de selección espacial *maximin*
con una distancia mínima de 50 km.

Para cada estado se selecciona inicialmente la estación con
mayor duración potencial de registros.

Posteriormente, se selecciona iterativamente la estación
cuya distancia mínima a las estaciones ya elegidas sea
máxima, hasta que ninguna candidata cumpla la separación
establecida.

El procedimiento reduce la concentración espacial de
estaciones, aunque no garantiza una cobertura temporal
completa ni una separación mínima entre estados diferentes.

In [8]:
df_seleccionadas = seleccionar_estaciones_espaciales(
    df_continentales,
    distancia_min_km=50,
)

print("Estaciones continentales:", len(df_continentales))
print("Estaciones seleccionadas:", len(df_seleccionadas))

display(
    df_seleccionadas.groupby("estado")
    .agg(estaciones=("estacion", "nunique"))
    .reset_index()
    .sort_values("estaciones", ascending=False)
)

2026-10-27 14:40:23.355 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - AGUASCALIENTES: 68 candidatas → 3 seleccionadas.
2026-10-27 14:40:23.365 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - BAJA CALIFORNIA: 133 candidatas → 22 seleccionadas.
2026-10-27 14:40:23.371 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - BAJA CALIFORNIA SUR: 157 candidatas → 21 seleccionadas.
2026-10-27 14:40:23.376 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - CAMPECHE: 73 candidatas → 15 seleccionadas.
2026-10-27 14:40:23.381 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - CHIAPAS: 266 candidatas → 19 seleccionadas.
2026-10-27 14:40:23.387 | INFO     | proyecto_agronomia.datasets.clima_con:seleccionar_estaciones_espaciales:700 - CHIHUAHUA: 227 candidatas → 45 seleccionadas.
2026-10-27 14:40:23

,estado,estaciones
5,CHIHUAHUA,45
25,SONORA,39
19,OAXACA,30
9,DURANGO,27
13,JALISCO,26
7,COAHUILA DE ZARAGOZA,25
29,VERACRUZ DE IGNACIO DE LA LLAVE,23
1,BAJA CALIFORNIA,22
31,ZACATECAS,21
2,BAJA CALIFORNIA SUR,21


## 5. Distribución geográfica de estaciones seleccionadas

Se representa la ubicación de las 531 estaciones seleccionadas
mediante un mapa interactivo.

La visualización permite inspeccionar la distribución espacial,
identificar regiones con menor cobertura y comprobar la exclusión
de estaciones insulares.

La selección espacial se realizó independientemente para cada
entidad federativa, utilizando una separación mínima de 50 km.

In [9]:


mapa = crear_mapa_estaciones_conagua(
    df_seleccionadas
)

display(mapa)

2026-10-27 14:40:23.652 | SUCCESS  | proyecto_agronomia.datasets.clima_con:crear_mapa_estaciones_conagua:776 - Mapa generado con 531 estaciones.


In [10]:
CONAGUA_PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ruta_mapa = (
    CONAGUA_PROCESSED_DIR
    / "mapa_estaciones_conagua.html"
)

mapa.save(str(ruta_mapa))

print("Mapa guardado:", ruta_mapa)

Mapa guardado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\clima\conagua\mapa_estaciones_conagua.html


## 6. Descarga nacional de datos climatológicos

Se descargan los archivos mensuales originales del Servicio
Meteorológico Nacional (CONAGUA) correspondientes a las
531 estaciones seleccionadas.

Los archivos se organizan por entidad federativa utilizando
las abreviaturas incluidas en las URL oficiales.

Se conservan los archivos TXT originales para garantizar
la trazabilidad y reproducibilidad de los datos.

La descarga es incremental: los archivos existentes no
se descargan nuevamente.

In [11]:
ruta_catalogo = (
    CONAGUA_PROCESSED_DIR
    / "estaciones_seleccionadas_continentales_1981_2022.csv"
)

df_seleccionadas.to_csv(
    ruta_catalogo,
    index=False,
    encoding="utf-8-sig",
)

print("Catálogo guardado:", ruta_catalogo)
print("Estaciones:", len(df_seleccionadas))

Catálogo guardado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\clima\conagua\estaciones_seleccionadas_continentales_1981_2022.csv
Estaciones: 531


In [12]:
archivos_descargados = descargar_estaciones_conagua(
    estaciones=df_seleccionadas,
    pausa=1.0,
)

print(
    "Archivos disponibles:",
    len(archivos_descargados)
)

Estaciones CONAGUA:  61%|██████    | 324/531 [00:01<00:00, 1052.61it/s]

2026-10-27 14:40:27.047 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [21114] Formato no reconocido.


Estaciones CONAGUA:  61%|██████    | 324/531 [00:03<00:00, 1052.61it/s]

2026-10-27 14:40:29.501 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [23025] Formato no reconocido.


Estaciones CONAGUA:  81%|████████  | 430/531 [00:06<00:01, 56.18it/s]  

...

2026-10-27 14:40:34.008 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [30353] Formato no reconocido.


Estaciones CONAGUA:  93%|█████████▎| 495/531 [00:10<00:00, 44.72it/s]

2026-10-27 14:40:35.954 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [31044] Formato no reconocido.


Estaciones CONAGUA: 100%|██████████| 531/531 [00:11<00:00, 46.62it/s]

2026-10-27 14:40:36.985 | SUCCESS  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:278 - Archivos CONAGUA disponibles: 526 de 531.
Archivos disponibles: 526


In [13]:
rutas_txt = sorted(
    CONAGUA_RAW_DIR.rglob("mes*.txt")
)

print("TXT encontrados:", len(rutas_txt))
print("Estaciones seleccionadas:", len(df_seleccionadas))

print("\nPrimeros archivos:")
for ruta in rutas_txt[:10]:
    print(ruta.relative_to(CONAGUA_RAW_DIR))

TXT encontrados: 526
Estaciones seleccionadas: 531

Primeros archivos:
ags\mes01001.txt
ags\mes01078.txt
ags\mes01088.txt
bc\mes02004.txt
bc\mes02022.txt
bc\mes02031.txt
bc\mes02037.txt
bc\mes02039.txt
bc\mes02040.txt
bc\mes02050.txt


In [14]:
from pathlib import Path
from urllib.parse import urlparse

from proyecto_agronomia.config import CONAGUA_RAW_DIR

def obtener_ruta_txt(url):
    ruta = Path(urlparse(url).path)

    return (
        CONAGUA_RAW_DIR
        / ruta.parent.name
        / ruta.name
    )

df_revision = df_seleccionadas.copy()

df_revision["ruta_txt"] = (
    df_revision["url_mensual"]
    .apply(obtener_ruta_txt)
)

df_revision["descargado"] = (
    df_revision["ruta_txt"]
    .apply(lambda ruta: ruta.is_file())
)

df_faltantes = df_revision[
    ~df_revision["descargado"]
].copy()

print("Estaciones seleccionadas:", len(df_revision))
print("Archivos encontrados:", df_revision["descargado"].sum())
print("Archivos faltantes:", len(df_faltantes))

display(
    df_faltantes[
        [
            "estacion",
            "nombre",
            "estado",
            "url_mensual",
        ]
    ]
)

Estaciones seleccionadas: 531
Archivos encontrados: 526
Archivos faltantes: 5


,estacion,nombre,estado,url_mensual
338,21114,ZOQUITLAN,PUEBLA,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
358,23025,TULUM,QUINTANA ROO,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
468,28147,TAMESI,TAMAULIPAS,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
495,30353,LA MANCHA,VERACRUZ DE IGNACIO DE LA LLAVE,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...
506,31044,MERIDA (OBS),YUCATÁN,https://smn.conagua.gob.mx/tools/RESOURCES/Nor...


In [15]:
archivos_recuperados = descargar_estaciones_conagua(
    estaciones=df_faltantes,
    pausa=1.0,
)

print(
    "Archivos disponibles tras el reintento:",
    len(archivos_recuperados)
)

Estaciones CONAGUA:   0%|          | 0/5 [00:01<?, ?it/s]

2026-10-27 14:40:38.410 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [21114] Formato no reconocido.


Estaciones CONAGUA:  20%|██        | 1/5 [00:03<00:08,  2.19s/it]

2026-10-27 14:40:40.570 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [23025] Formato no reconocido.


Estaciones CONAGUA:  40%|████      | 2/5 [00:05<00:06,  2.17s/it]

...

2026-10-27 14:40:44.789 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [30353] Formato no reconocido.


Estaciones CONAGUA:  80%|████████  | 4/5 [00:09<00:02,  2.13s/it]

2026-10-27 14:40:46.811 | WARNING  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:252 - [31044] Formato no reconocido.


Estaciones CONAGUA: 100%|██████████| 5/5 [00:10<00:00,  2.12s/it]

2026-10-27 14:40:47.819 | SUCCESS  | proyecto_agronomia.datasets.clima_con:descargar_estaciones_conagua:278 - Archivos CONAGUA disponibles: 0 de 5.
Archivos disponibles tras el reintento: 0


In [16]:
df_revision["descargado"] = (
    df_revision["ruta_txt"]
    .apply(lambda ruta: ruta.is_file())
)

print(
    "Archivos disponibles:",
    df_revision["descargado"].sum()
)

print(
    "Archivos faltantes:",
    (~df_revision["descargado"]).sum()
)

Archivos disponibles: 526
Archivos faltantes: 5


In [17]:

from collections import Counter

from proyecto_agronomia.config import CONAGUA_RAW_DIR

# Todos los TXT realmente almacenados
archivos_reales = list(
    CONAGUA_RAW_DIR.rglob("*.txt")
)

# Rutas esperadas de las estaciones seleccionadas
rutas_esperadas = set(
    df_revision["ruta_txt"]
)

# Archivos existentes que no coinciden con ninguna ruta esperada
archivos_no_coincidentes = [
    ruta
    for ruta in archivos_reales
    if ruta not in rutas_esperadas
]

print("TXT reales:", len(archivos_reales))
print("Rutas esperadas:", len(rutas_esperadas))
print(
    "Archivos en rutas distintas:",
    len(archivos_no_coincidentes)
)

print("\nCarpetas encontradas:")
for carpeta, cantidad in Counter(
    ruta.parent.name
    for ruta in archivos_reales
).most_common():
    print(f"{carpeta}: {cantidad}")

print("\nPrimeros archivos no coincidentes:")
for ruta in archivos_no_coincidentes[:15]:
    print(ruta.relative_to(CONAGUA_RAW_DIR))

TXT reales: 526
Rutas esperadas: 531
Archivos en rutas distintas: 0

Carpetas encontradas:
chih: 45
son: 39
oax: 30
dgo: 27
jal: 26
coah: 25
bc: 22
ver: 22
bcs: 21
zac: 21
gro: 20
slp: 20
tamps: 20
chis: 19
mich: 18
nl: 18
sin: 17
camp: 15
pue: 12
qroo: 12
mex: 11
yuc: 11
gto: 10
nay: 10
hgo: 9
tab: 8
qro: 5
ags: 3
col: 3
mor: 3
tlax: 3
df: 1

Primeros archivos no coincidentes:


In [18]:
# Índice de archivos existentes por nombre
indice_archivos = {}

for ruta in archivos_reales:
    indice_archivos.setdefault(
        ruta.name.lower(), []
    ).append(ruta)

df_faltantes = df_revision[
    ~df_revision["descargado"]
].copy()

df_faltantes["archivo_esperado"] = (
    df_faltantes["ruta_txt"]
    .apply(lambda ruta: ruta.name.lower())
)

df_faltantes["en_otra_carpeta"] = (
    df_faltantes["archivo_esperado"]
    .apply(lambda nombre: nombre in indice_archivos)
)

print(
    "Faltantes encontrados en otra carpeta:",
    df_faltantes["en_otra_carpeta"].sum()
)

print(
    "Faltantes sin archivo coincidente:",
    (~df_faltantes["en_otra_carpeta"]).sum()
)

display(
    df_faltantes[
        [
            "estacion",
            "nombre",
            "estado",
            "archivo_esperado",
            "en_otra_carpeta",
        ]
    ]
)

Faltantes encontrados en otra carpeta: 0
Faltantes sin archivo coincidente: 5


,estacion,nombre,estado,archivo_esperado,en_otra_carpeta
338,21114,ZOQUITLAN,PUEBLA,mes21114.txt,False
358,23025,TULUM,QUINTANA ROO,mes23025.txt,False
468,28147,TAMESI,TAMAULIPAS,mes28147.txt,False
495,30353,LA MANCHA,VERACRUZ DE IGNACIO DE LA LLAVE,mes30353.txt,False
506,31044,MERIDA (OBS),YUCATÁN,mes31044.txt,False


In [19]:
estacion_prueba = df_revision[
    df_revision["estacion"].astype(str) == "1001"
]

for _, fila in estacion_prueba.iterrows():
    print("Estación:", fila["estacion"])
    print("URL:", fila["url_mensual"])
    print("Ruta esperada:", fila["ruta_txt"])
    print("Existe:", fila["ruta_txt"].exists())

print("\nArchivos similares encontrados:")

for ruta in CONAGUA_RAW_DIR.rglob("*1001*.txt"):
    print(ruta)

Estación: 1001
URL: https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Mensuales/ags/mes01001.txt
Ruta esperada: B:\MCD2\JW\agro\proyecto_agronomia\data\raw\clima\conagua\ags\mes01001.txt
Existe: True

Archivos similares encontrados:
B:\MCD2\JW\agro\proyecto_agronomia\data\raw\clima\conagua\ags\mes01001.txt


# Pasar a json

In [20]:
resumen = convertir_conagua_nacional()

print("TXT encontrados:", resumen["total_txt"])
print("JSON convertidos:", resumen["convertidos"])
print("JSON existentes:", resumen["existentes"])
print("Errores:", resumen["errores"])

2026-10-27 14:40:48.135 | INFO     | proyecto_agronomia.datasets.clima_con:convertir_conagua_nacional:981 - Se encontraron 526 archivos TXT.


Convirtiendo CONAGUA a JSON:   0%|          | 0/526 [00:00<?, ?it/s]

Convirtiendo CONAGUA a JSON: 100%|██████████| 526/526 [00:00<00:00, 2198.09it/s]

2026-10-27 14:40:48.380 | SUCCESS  | proyecto_agronomia.datasets.clima_con:convertir_conagua_nacional:1034 - Procesamiento CONAGUA terminado. Convertidos: 0 | Existentes: 526 | Errores: 0
TXT encontrados: 526
JSON convertidos: 0
JSON existentes: 526
Errores: 0
